In [1]:
import matplotlib.pyplot as plt

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def get_logs_data(logdir, tag_name):
    """Wyciąga dane dla konkretnego tagu ze wszystkich podfolderów w logdir."""
    all_runs_data = []
    run_folders = [os.path.join(logdir, d) for d in os.listdir(logdir) 
                   if os.path.isdir(os.path.join(logdir, d))]
    run_folders = sorted(run_folders)[:10]
    print(f"Znaleziono {len(run_folders)} folderów z logami.")
    for run_path in run_folders:
        print(f"Przetwarzanie: {run_path}...")
        event_acc = EventAccumulator(run_path)
        event_acc.Reload()
        if tag_name in event_acc.Tags()['scalars']:
            events = event_acc.Scalars(tag_name)
            steps = [e.step for e in events]
            values = [e.value for e in events]
            all_runs_data.append(pd.Series(values, index=steps))
        else:
            print(f"  Ostrzeżenie: Brak tagu '{tag_name}' w {run_path}")
    return all_runs_data

def plot_with_std(logdir, tag_name):
    data_series = get_logs_data(logdir, tag_name)
    
    if not data_series:
        print("Nie znaleziono żadnych danych do narysowania.")
        return
    df = pd.concat(data_series, axis=1)
    mean = df.mean(axis=1)
    std = df.std(axis=1)
    steps = df.index

    plt.figure(figsize=(10, 6))
    plt.plot(steps, mean, label='Średnia', color='blue', linewidth=2)
    plt.fill_between(steps, mean - std, mean + std, color='blue', alpha=0.2, label='Odchylenie standardowe')

    plt.title(f'Wykres: {tag_name} (Średnia ± STD z 10 przebiegów)')
    plt.xlabel('Krok (Step)')
    plt.ylabel('Wartość')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    
    plt.savefig('wykres_odchylenia.png')
    print("Wykres został zapisany jako 'wykres_odchylenia.png'")
    plt.show()

PATH_TO_LOGS = 'tb_logs/' 
TAG_TO_PLOT = 'epoch_loss' 

plot_with_std(PATH_TO_LOGS, TAG_TO_PLOT)